In [2]:
import math
import json
import numpy as np
import pandas as pd
from typing import Tuple, List

In [3]:
# finds divisors of integer n
def find_divisors(n):
    divisors = []
    for i in range(1,n+1):
        if n % i == 0:
            divisors.append(i)
    return divisors

# produces a standard tableaux with shape ls
def stab(ls):
    stab = []
    i =1
    for l in ls:
        row = list(range(i,l+i))
        i += l
        stab.append(row)
    return stab


# Produces Young diagrams with corresponding removable boxes such that removing the box from one Young diagram yields the Young diagram corresponding to ls 
def up_diagrams(ls): 
    diagrams = []
    ups = []
    lsnew = list(ls)
    lsnew[0] += 1
    diagrams.append(lsnew)
    ups.append((1,lsnew[0]))
    row_length = ls[0]
    for index, row in enumerate(ls):
        if row < row_length:
            lsnew = list(ls)
            lsnew[index] += 1
            diagrams.append(lsnew)
            ups.append((index+1,row+1))
            row_length = row
    lsnew = list(ls)
    lsnew.append(1)
    diagrams.append(lsnew) 
    ups.append((len(ls)+1,1))
    return diagrams, ups

# for a Young diagram of shape ls, lists removable boxes
def removables(ls): 
    removs = []
    length = len(ls)
    row_length = ls[0]
    for index, row in enumerate(ls):
        if row < row_length:
            removs.append((index, ls[index-1]))
        row_length = row
    removs.append((length, ls[length-1]))
    return removs

# given a list representing a Young diagram and two boxes in the Young diagram computes the axial distance between them 
def axial_dist(ls: list, a: Tuple[int,int], b: Tuple[int, int]):
    if (a[1] <= ls[a[0]-1]) and (b[1] <=ls[b[0]-1]):
        return (a[1]-a[0])- (b[1]-b[0])
    else:
        raise ValueError("Boxes not within Young diagram")
    
# for a Young diagram of shape ls produces a tableau of shape ls in which every box contains its own hook length
def hook(ls):
    s_tab = stab(ls)
    hooks = s_tab
    nrows = len(s_tab)
    for row_index, row in enumerate(s_tab):
        for index in range(len(row)):
            hook = len(row) - index
            row_index2 = row_index +1
            while row_index2 < nrows:
                if (len(s_tab[row_index2]) > index):
                    hook += 1
                row_index2 += 1
            hooks[row_index][index] = hook
    return hooks

# for a Young diagram of shape ls, produces the corresponding hook length (dimension of the corresponding representation of S_n)
def hook_length(ls):
    part = sum(ls)
    prod = 1
    hooks = hook(ls)
    for row in hooks:
        prod = prod * math.prod(row)
    return math.factorial(part)//prod

# given an integer n produces all partitions of n
def generate_partitions(n):
    partitions = []
    if n == 0:
        return [[]]
    for i in range(0,n):
        for p in generate_partitions(i):
                if p == [] or max(p) <= n-i: 
                    partitions.append([n-i] + p)
    return partitions

# need to fix a mu in mus and then condsider removs of mu to obtain q
def beta(num: int, mu: List[int], L: List[List[int]], upremov: List[Tuple]):
    betas = []
    qs = removables(mu)
    nqs = len(qs)
    for q in range(nqs):
        dim_axial = 0
        for i in range(0, len(L)):
            #print(axial_dist(L[i],upremov[i], qs[q]))
            dim_axial += hook_length(L[i])/axial_dist(L[i],upremov[i], qs[q])            
        betas.append((-1)**q/(num * hook_length(mu)) * dim_axial)
    return betas

def constant_list(ls):
    val = True
    for l in ls:
        val = val and (l == ls[0])
    return val

In [ ]:
# accelerated algorithm for obtaining descending partitions (arXiv:0909.2331v2 [cs.DS])
def accel_desc(n):
    if n < 1:
        raise ValueError("Integer must be at least 1 to generate partition")
    k = 1
    q = 1
    d = [1 for i in range(n)]
    d[0] = n
    yield [d[0]]
    while q != 0:
        if d[q-1] == 2:
            k = k+1
            d[q-1] = 1
            q = q-1
        else:
            m = d[q-1]-1
            n_ = k-q+1
            d[q-1] = m
            while n_ >= m:
                q = q+1
                d[q-1] = m
                n_ = n_ - m
            if n_ == 0:
                k = q
            else:
                k = q+1
                if n_ > 1:
                    q = q+1
                    d[q-1] = n_
        yield d[:k]

In [5]:
with open("EITFFs_L0s.txt", 'w') as fileL0:
    fileL0.write("EITFF parameters | Partition of n^2-1 (mu) | L0 Up-diagrams | betas for removables of mu \n")
with open("EITFFs_L1s.txt", 'w') as fileL1:
    fileL1.write("EITFF parameters | Partition of n^2-1 (mu) | L1 Up-diagrams | betas for removables of mu \n")
for m in range(3,4):
    n=m*m
    mus = list(accel_desc(n-1)) # partitions of n-1
    nmus= len(mus) # number of paritions of n-1
    removs = [removables(mu) for mu in mus] # list containing a list of removable boxes for each partition of n-1
    ups = [up_diagrams(mu) for mu in mus] # list containing all partitions of n that arise from a partition of n-1
    L0s = [ups[i][0][1::2] for i in range(nmus)] # list containing L0 (list of Young diagrams for n) for each partition of n-1
    up0removs = [ups[i][1][1::2] for i in range(nmus)] # list containing a list of a list of removable boxes for each partition in L0 for each partition of n-1
    L1s = [ups[i][0][0::2] for i in range(nmus)] # list containing L0 (list of Young diagrams for n) for each partition of n-1
    up1removs = [ups[i][1][0::2] for i in range(nmus)] # list containing a list of a list of removable boxes for each partition in L1 for each partition of n-1
    dL0s = [sum(map(hook_length, L0)) for L0 in L0s] # list containing dimension of V_{L0} for each L0 in L0s
    dL1s = [sum(map(hook_length, L1)) for L1 in L1s] # list containing dimension of V_{L1} for each L1 in L1s
    beta0s = [beta(n, mus[i], L0s[i], up0removs[i]) for i in range(nmus)]
    beta1s = [beta(n, mus[i], L1s[i], up1removs[i]) for i in range(nmus)]
    for i in range(nmus):
        dL0 = dL0s[i]
        dmu = hook_length(mus[i])
        if constant_list(beta0s[i]) and (dL0 == dmu*m):
            print((dL0, dmu, n), mus[i], L0s[i], beta0s[i])
            with open("EITFFs_L0s.txt", 'a') as fileL0:
                json.dump([(dL0, dmu, n), mus[i], L0s[i], beta0s[i]], fileL0, indent="\t")
    for i in range(nmus):
        dL1 = dL1s[i]
        dmu = hook_length(mus[i])
        if constant_list(beta1s[i]) and (dL1 == dmu*m):
            print((dL1, dmu, n), mus[i], L1s[i], beta1s[i])
            with open("EITFFs_L1s.txt", 'a') as fileL1:
                
                json.dump([(dL1, dmu, n), mus[i], L1s[i], beta1s[i]],fileL1, indent="\t")

(168, 56, 9) [4, 2, 2] [[4, 3, 2]] [-0.16666666666666666, -0.16666666666666666]
(168, 56, 9) [3, 3, 1, 1] [[3, 3, 2, 1]] [-0.16666666666666666, -0.16666666666666666]
(42, 14, 9) [2, 2, 2, 2] [[2, 2, 2, 2, 1]] [-0.16666666666666666]
(42, 14, 9) [4, 4] [[5, 4]] [0.16666666666666666]


In [7]:
df = pd.read_csv('EITFFs_L0s.txt', sep='|', nrows=0)
df

,EITFF parameters,Partition of n^2-1 (mu),L0 Up-diagrams,betas for removables of mu


In [8]:
with open('EITFFs_L0s.txt','r') as f:
    lines = f.readlines()
    ls = []
    new_column= False
    row = []
    column = []
    for line in lines[1:]:
        if (line == ']['):
            print(row)
            ls.append(row)
            row = []
        if (line == '	],'):
            print(column)
            row.append(column)
            column = []
            new_column = False
        if (line == '['):
            new_column = True
        if (line == '	['):
            new_column = True
        if new_column & (line != '		[') & (line != '		]'):
            print(float(line.strip()))
            column.append(float(line.strip()))

In [9]:
def is_good(diagram: list):
    head = diagram[0]
    tail = 0
    bump = 0
    for row in diagram:
        if row == head:
            continue
        else:
            bump = row
            tail += 1
    if head - bump == tail:
        c = tail
        a = len(diagram)- c
        b = head - c
        if ((a*b) % c == 0) & ((a+c)*(b+c) == c**2 * (c**2-1)):
            return True
    else:
        return False
    
ls = [[4,2,2], [3,3,1,1], [18,15,15,15], [12,12,12,9,9,9],[9,9,9,9,9,6,6,6],[8,8,8,8,8,8,5,5,5],[6,6,6,6,6,6,6,6,6,3,3,3], [4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,1,1,1]]
for diagram in ls:
    print(is_good(diagram))

True
True
True
True
True
True
True
True
